In [1]:
# ─────────────────────────────────────────────────────────────────
# Cell 1 — Configuration
# ─────────────────────────────────────────────────────────────────
from pathlib import Path

CELEBA_ROOT = "./data/celeba"
SPLIT       = "train"   # "train", "val", "test", or "all"

In [2]:
# ─────────────────────────────────────────────────────────────────
# Cell 2 — Load all 40 attributes for the split
# ─────────────────────────────────────────────────────────────────
import numpy as np
import torch
from pathlib import Path

def load_all_attributes(celeba_root, split="train"):
    """
    Returns:
        attr_matrix : np.ndarray [N_images, 40]  values in {0, 1}
        attr_names  : list of 40 attribute name strings
    """
    attr_file = Path(celeba_root) / "list_attr_celeba.txt"
    with open(attr_file) as f:
        lines = f.readlines()

    attr_names = lines[1].strip().split()   # 40 names

    if split == "all":
        # use every image
        data_lines = lines[2:]
    else:
        partition_file = Path(celeba_root) / "list_eval_partition.txt"
        split_map = {"train": 0, "val": 1, "test": 2}
        target = split_map[split]
        with open(partition_file) as f:
            partitions = [int(l.strip().split()[-1]) for l in f if l.strip()]
        data_lines = [
            lines[2 + img_idx]
            for img_idx, part in enumerate(partitions)
            if part == target
        ]

    attr_matrix = np.array([
        [1 if int(v) == 1 else 0 for v in line.strip().split()[1:]]   # skip filename
        for line in data_lines
    ], dtype=np.float32)

    print(f"Loaded {attr_matrix.shape[0]} images × {attr_matrix.shape[1]} attributes ({split} split)")
    print(f"Overall positive rate: {attr_matrix.mean():.3f}")
    return attr_matrix, attr_names


attr_matrix, attr_names = load_all_attributes(CELEBA_ROOT, split=SPLIT)

Loaded 162770 images × 40 attributes (train split)
Overall positive rate: 0.226


In [3]:
# ─────────────────────────────────────────────────────────────────
# Cell 3 — Compute co-occurrence and correlation matrices
# ─────────────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(attr_matrix, columns=attr_names)

# ── 1. Pearson correlation between binary attribute vectors ───────
corr = df.corr()                           # [40 × 40]

# ── 2. Conditional probability P(attr_j=1 | attr_i=1) ────────────
# entry [i, j] = P(j | i) — how often j is positive given i is positive
n_attrs = len(attr_names)
cond_prob = np.zeros((n_attrs, n_attrs))
for i in range(n_attrs):
    mask = attr_matrix[:, i] == 1
    if mask.sum() > 0:
        cond_prob[i] = attr_matrix[mask].mean(axis=0)

cond_prob_df = pd.DataFrame(cond_prob, index=attr_names, columns=attr_names)

# ── 3. Base rates (marginal P(attr=1)) ────────────────────────────
base_rates = df.mean().sort_values(ascending=False)
print("Attribute base rates:")
print(base_rates.to_string())

Attribute base rates:
No_Beard               0.834177
Young                  0.778940
Attractive             0.513627
Mouth_Slightly_Open    0.482190
Smiling                0.479695
Wearing_Lipstick       0.469601
High_Cheekbones        0.452448
Male                   0.419371
Heavy_Makeup           0.384315
Wavy_Hair              0.319359
Oval_Face              0.283228
Pointy_Nose            0.275518
Arched_Eyebrows        0.265884
Big_Lips               0.240910
Black_Hair             0.239024
Big_Nose               0.235553
Straight_Hair          0.208558
Bags_Under_Eyes        0.204460
Brown_Hair             0.203920
Wearing_Earrings       0.186533
Bangs                  0.151656
Blond_Hair             0.149088
Bushy_Eyebrows         0.143675
Wearing_Necklace       0.121423
Narrow_Eyes            0.115924
5_o_Clock_Shadow       0.111673
Receding_Hairline      0.080113
Wearing_Necktie        0.073048
Rosy_Cheeks            0.064662
Eyeglasses             0.064637
Goatee            

In [5]:
# ─────────────────────────────────────────────────────────────────
# Cell 4 — Sort attributes by hierarchical clustering
#           so related attributes appear next to each other
# ─────────────────────────────────────────────────────────────────
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

# Convert correlation to distance (1 - |corr|) for clustering
dist = 1 - corr.abs().values
np.fill_diagonal(dist, 0)
dist = np.clip(dist, 0, None)   # numerical safety

linkage_matrix = linkage(squareform(dist), method="ward")
order = leaves_list(linkage_matrix)

attr_names_ordered = [attr_names[i] for i in order]

# ── FIX: use .iloc for both row and column reordering ────────────
corr_ordered     = corr.iloc[order, :].iloc[:, order]
cond_prob_df     = pd.DataFrame(cond_prob, index=attr_names, columns=attr_names)
cond_ordered     = cond_prob_df.iloc[order, :].iloc[:, order]

In [6]:
# ─────────────────────────────────────────────────────────────────
# Cell 5 — Plot 1: Pearson correlation heatmap
# ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

fig, ax = plt.subplots(figsize=(16, 14))

sns.heatmap(
    corr_ordered,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-0.6, vmax=0.6,
    square=True,
    linewidths=0.3,
    linecolor="#333333",
    annot=False,
    cbar_kws={"label": "Pearson r", "shrink": 0.8},
)

ax.set_title(
    f"CelebA attribute correlations ({SPLIT} split, N={attr_matrix.shape[0]:,})",
    fontsize=15, fontweight="bold", pad=14,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig(f"celeba_corr_heatmap_{SPLIT}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → celeba_corr_heatmap_{SPLIT}.png")

Saved → celeba_corr_heatmap_train.png


In [7]:
# ─────────────────────────────────────────────────────────────────
# Cell 6 — Plot 2: Conditional probability heatmap
#           P(col | row) — read row → col
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 14))

sns.heatmap(
    cond_ordered,
    ax=ax,
    cmap="YlOrRd",
    vmin=0, vmax=1,
    square=True,
    linewidths=0.3,
    linecolor="#333333",
    annot=False,
    cbar_kws={"label": "P(col = 1 | row = 1)", "shrink": 0.8},
)

ax.set_title(
    f"CelebA conditional probabilities  P(col | row)  —  {SPLIT} split",
    fontsize=15, fontweight="bold", pad=14,
)
ax.set_xlabel("Given this attribute is 1 →", fontsize=10)
ax.set_ylabel("← probability this attribute is 1", fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig(f"celeba_condprob_heatmap_{SPLIT}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → celeba_condprob_heatmap_{SPLIT}.png")

Saved → celeba_condprob_heatmap_train.png


In [8]:
# ─────────────────────────────────────────────────────────────────
# Cell 7 — Plot 3: Base rate bar chart
#           Shows how imbalanced each attribute is
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

colors = ["#e74c3c" if r < 0.1 or r > 0.9 else "#3498db" for r in base_rates.values]
bars = ax.bar(range(len(base_rates)), base_rates.values, color=colors, edgecolor="white", linewidth=0.5)

ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="50% base rate")
ax.axhspan(0, 0.1,  alpha=0.05, color="red",  label="<10% or >90% (highly imbalanced)")
ax.axhspan(0.9, 1.0, alpha=0.05, color="red")

ax.set_xticks(range(len(base_rates)))
ax.set_xticklabels(base_rates.index, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("P(attribute = 1)", fontsize=11)
ax.set_title(
    f"CelebA attribute base rates ({SPLIT} split, N={attr_matrix.shape[0]:,})\n"
    f"Red bars = highly imbalanced (<10% or >90%)",
    fontsize=13, fontweight="bold",
)
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"celeba_base_rates_{SPLIT}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → celeba_base_rates_{SPLIT}.png")

Saved → celeba_base_rates_train.png


In [9]:
# ─────────────────────────────────────────────────────────────────
# Cell 8 — Print strongest biases in plain text
#           Useful to quickly see what the heatmap shows
# ─────────────────────────────────────────────────────────────────

print("=" * 65)
print("TOP 20 STRONGEST ATTRIBUTE CORRELATIONS (Pearson |r|)")
print("=" * 65)
corr_vals = corr.abs()
np.fill_diagonal(corr_vals.values, 0)   # ignore self-correlation
pairs = (
    corr_vals.stack()
    .reset_index()
    .rename(columns={"level_0": "attr_a", "level_1": "attr_b", 0: "r"})
    .query("attr_a < attr_b")            # avoid duplicates
    .sort_values("r", ascending=False)
    .head(20)
)
for _, row in pairs.iterrows():
    sign = "+" if corr.loc[row.attr_a, row.attr_b] > 0 else "−"
    print(f"  {sign}  {row.attr_a:<25} ↔  {row.attr_b:<25}  r={corr.loc[row.attr_a, row.attr_b]:+.3f}")

print()
print("=" * 65)
print("TOP 15 STRONGEST CONDITIONAL PROBABILITIES P(col | row)")
print("=" * 65)
np.fill_diagonal(cond_prob, 0)
cond_df2 = pd.DataFrame(cond_prob, index=attr_names, columns=attr_names)
cond_pairs = (
    cond_df2.stack()
    .reset_index()
    .rename(columns={"level_0": "given", "level_1": "then", 0: "p"})
    .query("given != then")
    .sort_values("p", ascending=False)
    .head(15)
)
for _, row in cond_pairs.iterrows():
    base = df[row["then"]].mean()
    lift = row["p"] / base if base > 0 else float("inf")
    print(f"  P({row['then']:<22} | {row['given']:<22}) = {row['p']:.3f}  (base={base:.3f}, lift={lift:.1f}x)")

TOP 20 STRONGEST ATTRIBUTE CORRELATIONS (Pearson |r|)
  +  Heavy_Makeup              ↔  Wearing_Lipstick           r=+0.802
  −  Male                      ↔  Wearing_Lipstick           r=-0.789
  +  High_Cheekbones           ↔  Smiling                    r=+0.683
  −  Heavy_Makeup              ↔  Male                       r=-0.666
  −  Goatee                    ↔  No_Beard                   r=-0.571
  −  No_Beard                  ↔  Sideburns                  r=-0.540
  +  Mouth_Slightly_Open       ↔  Smiling                    r=+0.535
  +  Chubby                    ↔  Double_Chin                r=+0.530
  −  5_o_Clock_Shadow          ↔  No_Beard                   r=-0.526
  −  Male                      ↔  No_Beard                   r=-0.521
  +  Goatee                    ↔  Sideburns                  r=+0.512
  +  Attractive                ↔  Wearing_Lipstick           r=+0.485
  +  Attractive                ↔  Heavy_Makeup               r=+0.481
  +  Arched_Eyebrows           ↔  We